In [1]:
import pandas as pd
import requests # type: ignore

In [2]:
# System and OS related tasks
import sys
import os

# Add the project root to Python path
project_root = os.path.abspath('..')
sys.path.insert(0, project_root)

# path to directories
raw_dir = '../data/raw'
processed_dir = '../data/processed'

# 1.0 ETL For World Bank's Precipitation Data 

## 1.1 Get the JSON fom World Bank Climate Change Knowledge Portal (CCKP)
📌 The following parameters were used to generate both temperature data and the precipitation data for all countries:
* Area of Focus: Global Countries (global_countries)
* Collection: CRU 0.5-degree (cru-x0.5)
  * The World Bank's CCKP uses the 0.5-degree resolution CRU dataset because it represents a unique combination of long-term historical coverage, reliability for trend analysis, and suitability for national-scale assessments
* Type: Timeseries (timeseries)
* Variable: Average Mean Surface Air Temperature (`tas`) or precipitation (`pr`)
* Product: Time Series (timeseries)
* Aggregation: Annual (annual)
* Period: 1901-2024

📌 **In this notebook, we will get the PRECIPITATION data of the 26 tea coffee producing countries.**

---

## 1.2 Get A List of Top 15 Tea and Coffee Producing Countries

📌 Let's extract the data for the top tea and coffee producing countries.

In [3]:
# Get a list of the top tea and coffee producing countries.
filename = "df_crop_countries.csv"

data_columns = ['country_ISO3', 'crop', 'country_name_ISO3']
column_dtypes = {
    'country_ISO3': str, 'crop': str, 'country_name_ISO3' : str}

df_crop_countries = pd.read_csv(
    f"{processed_dir}/{filename}",
    header=0,
    names=data_columns,
    dtype=column_dtypes
    )
df_crop_countries.head(3)

,country_ISO3,crop,country_name_ISO3
0,ARG,tea only,Argentina
1,BGD,tea only,Bangladesh
2,BRA,coffee only,Brazil


---

## 1.3 World Bank's Precipitation Data using the Climate Change Knowledge Portal (CCKP)


### 1.3.1 Get the Precipitation Data and Transformation to Long Format

In [4]:
url_precip = "https://cckpapi.worldbank.org/api/v1/cru-x0.5_timeseries_pr_timeseries_annual_1901-2024_mean_historical_cru_ts4.09_mean/global_countries?_format=json"

response_precip = requests.get(url_precip)
wb_precip_all_dict = response_precip.json() 

📌 Taking a look at the returned JSON:

In [ ]:
# take a look at the wb_precip_all_dict JSON
print("Type:", type(wb_precip_all_dict )) 
print("Top-level keys:", wb_precip_all_dict .keys())
print("Number of top-level items:", len(wb_precip_all_dict ))

Type: <class 'dict'>
Top-level keys: dict_keys(['metadata', 'data'])
Number of top-level items: 2


📌 It looks like there are 2 dictionaries inside with keys `metadata` and `data`.  Take peek into both of them:

In [6]:
# take a look at the metadata and data
print("Metadata:", wb_precip_all_dict.get('metadata', {}))
print("Data:", wb_precip_all_dict.get('data', {}))

Metadata: {'apiVersion': 'v1', 'status': 'success', 'messages': []}
Data: {'ABW': {'1901-07': 420.9, '1902-07': 420.9, '1903-07': 420.9, '1904-07': 420.9, '1905-07': 420.9, '1906-07': 420.9, '1907-07': 420.9, '1908-07': 420.9, '1909-07': 420.9, '1910-07': 420.9, '1911-07': 569.7, '1912-07': 487, '1913-07': 413.3, '1914-07': 393.8, '1915-07': 379.9, '1916-07': 442.3, '1917-07': 447.7, '1918-07': 434.3, '1919-07': 253.3, '1920-07': 324.1, '1921-07': 363.1, '1922-07': 503.1, '1923-07': 367.7, '1924-07': 574, '1925-07': 275.7, '1926-07': 446.4, '1927-07': 624.1, '1928-07': 486.3, '1929-07': 342.8, '1930-07': 305.5, '1931-07': 498.5, '1932-07': 475.7, '1933-07': 535.4, '1934-07': 439.2, '1935-07': 483.3, '1936-07': 455.9, '1937-07': 558.6, '1938-07': 544.8, '1939-07': 430.1, '1940-07': 288.4, '1941-07': 218.8, '1942-07': 620.7, '1943-07': 443.2, '1944-07': 479.4, '1945-07': 305.1, '1946-07': 378.8, '1947-07': 314.1, '1948-07': 310.7, '1949-07': 438.5, '1950-07': 574.5, '1951-07': 393, '1952

📌 It looks like the precipitation data is stored inside the data dictionary of the JSON with country code as the key. 

📌 The precipitation data actually goes back to year 1901 but we only need data for years 2000-2024.

📌 Now get the data dictionary out of the JSON stored in wb_precip_all_dict.

In [7]:
wb_precip_country_dict = wb_precip_all_dict['data']

print(f"Number of key-value pairs inside wb_precip_country_dict: {len(wb_precip_country_dict)}")

Number of key-value pairs inside wb_precip_country_dict: 246


📌 We only want weather Precipitation data for the 26 tea and coffee producing countries.

In [8]:
crop_country_precip_dict = {}

for country in df_crop_countries["country_ISO3"]:
    if country in wb_precip_country_dict:
        crop_country_precip_dict[country] = wb_precip_country_dict[country]
        
        print(f"A tea coffee crop country is found: {country}")
              
print(f"Number of records: {len(crop_country_precip_dict)}")
print(f"Look at China's years: {list(crop_country_precip_dict["CHN"])}")
print(f"Look at China's temperate: {crop_country_precip_dict["CHN"]["1901-07"]}")

A tea coffee crop country is found: ARG
A tea coffee crop country is found: BGD
A tea coffee crop country is found: BRA
A tea coffee crop country is found: CHN
A tea coffee crop country is found: CIV
A tea coffee crop country is found: COL
A tea coffee crop country is found: CRI
A tea coffee crop country is found: ETH
A tea coffee crop country is found: GIN
A tea coffee crop country is found: GTM
A tea coffee crop country is found: HND
A tea coffee crop country is found: IDN
A tea coffee crop country is found: IND
A tea coffee crop country is found: IRN
A tea coffee crop country is found: JPN
A tea coffee crop country is found: KEN
A tea coffee crop country is found: LKA
A tea coffee crop country is found: MEX
A tea coffee crop country is found: MWI
A tea coffee crop country is found: NIC
A tea coffee crop country is found: PER
A tea coffee crop country is found: RWA
A tea coffee crop country is found: TUR
A tea coffee crop country is found: TZA
A tea coffee crop country is found: UGA


📌 clean the year data which is the form of "1901-07" into a 4 digit year and then put the country, year, Precipitation data into long form dataframe.

In [9]:
crop_country_precip_list = []

# loop through to get the country and the {year: temp} out
for country, year_dict in crop_country_precip_dict.items():
    # loop through to get the year and temperature out
    for year_month, precip in year_dict.items():
        # get the year from "1901-07" format by splitting by "-" and take the left side where index = 0
        year = int(year_month.split("-")[0])

        # add to the list for data only for period 2000 to 2024
        if year >= 2000 and year <=2024:
            crop_country_precip_list.append({
                "country_ISO3":  country
                , "year": year
                , "precipitation": precip
            })

# convert crop_country_temp_list  into a dataframe
df_crop_country_precip = pd.DataFrame(crop_country_precip_list)        

df_crop_country_precip.head(5)


,country_ISO3,year,precipitation
0,ARG,2000,721.03
1,ARG,2001,716.55
2,ARG,2002,757.75
3,ARG,2003,593.14
4,ARG,2004,576.61


📌 Check if each country has Precipitation data for each of the years 2000-2024.

In [10]:
country_precip_stats = df_crop_country_precip.groupby("country_ISO3").agg(
    {"year": ["min", "max","count","nunique"]}
)

country_precip_stats

year                    
               min   max count nunique
country_ISO3                          
ARG           2000  2024    25      25
BGD           2000  2024    25      25
BRA           2000  2024    25      25
CHN           2000  2024    25      25
CIV           2000  2024    25      25
COL           2000  2024    25      25
CRI           2000  2024    25      25
ETH           2000  2024    25      25
GIN           2000  2024    25      25
GTM           2000  2024    25      25
HND           2000  2024    25      25
IDN           2000  2024    25      25
IND           2000  2024    25      25
IRN           2000  2024    25      25
JPN           2000  2024    25      25
KEN           2000  2024    25      25
LKA           2000  2024    25      25
MEX           2000  2024    25      25
MWI           2000  2024    25      25
NIC           2000  2024    25      25
PER           2000  2024    25      25
RWA           2000  2024    25      25
TUR           2000  2024    25      25
TZA           2000  2024    25      25
UGA           2000  2024    25      25
VNM           2000  2024    25      25

📌 Add the "crop" column to the precipitation file so that we know whether the country is tea or coffee only or both.

In [11]:
# create a mapping of country to crop types
country_crop_mapping = dict(zip(df_crop_countries["country_ISO3"], df_crop_countries["crop"]))

# add a new column "crop" to the temperature file 
df_crop_country_precip["crop"] = df_crop_country_precip["country_ISO3"].map(country_crop_mapping)

---

### 1.3.2 Export World Bank's Precipitation Data to CSV for Forward Analysis

In [12]:
df_crop_country_precip.to_csv(f'{processed_dir}/df_crop_country_precip.csv', index=False)
print(f"Exported: {processed_dir}/df_crop_country_precip.csv")

Exported: ../data/processed/df_crop_country_precip.csv


---

# 2.0 Merge the Temperature and Precipitation csvs into a Single csv

📌 Load the tea coffee country temperature csv file.

In [13]:
# Get a list of the top tea and coffee producing countries.
filename = "df_crop_country_temp.csv"

data_columns = ["country_ISO3", "year", "temperature", "crop"]
column_dtypes = {
    "country_ISO3": str, "crop": str, "temperature ": float, "crop": str}

df_crop_country_temp = pd.read_csv(
    f"{processed_dir}/{filename}",
    header=0,
    names=data_columns,
    dtype=column_dtypes
    )
df_crop_country_temp.head(3)

,country_ISO3,year,temperature,crop
0,ARG,2000,14.45,tea only
1,ARG,2001,15.08,tea only
2,ARG,2002,14.84,tea only


📌 Load the tea coffee country temperature and precipitation dataframes into 1 merged dataframes based on country-year. 

In [14]:
df_crop_country_climate = pd.merge(
    df_crop_country_temp
    , df_crop_country_precip
    , on=["country_ISO3","year"]
)

df_crop_country_climate.head(10)

,country_ISO3,year,temperature,crop_x,precipitation,crop_y
0,ARG,2000,14.45,tea only,721.03,tea only
1,ARG,2001,15.08,tea only,716.55,tea only
2,ARG,2002,14.84,tea only,757.75,tea only
3,ARG,2003,15.23,tea only,593.14,tea only
4,ARG,2004,15.24,tea only,576.61,tea only
5,ARG,2005,14.90,tea only,574.79,tea only
6,ARG,2006,15.46,tea only,597.45,tea only
7,ARG,2007,14.44,tea only,586.57,tea only
8,ARG,2008,15.33,tea only,535.56,tea only
9,ARG,2009,15.42,tea only,537.03,tea only


📌 Remove the extra crop column as that was the remnant of the dataframe merging.

In [15]:
# drop crop_x and keep crop_y
df_crop_country_climate = df_crop_country_climate.drop(columns=["crop_x"])
# rename crop_y to just crop
df_crop_country_climate = df_crop_country_climate.rename(columns={"crop_y": "crop"})

df_crop_country_climate.head(3)

,country_ISO3,year,temperature,precipitation,crop
0,ARG,2000,14.45,721.03,tea only
1,ARG,2001,15.08,716.55,tea only
2,ARG,2002,14.84,757.75,tea only


📌 check that there are no duplicates. 

In [16]:
df_crop_country_climate.value_counts("year")

year
2000    26
2001    26
2002    26
2003    26
2004    26
2005    26
2006    26
2007    26
2008    26
2009    26
2010    26
2011    26
2012    26
2013    26
2014    26
2015    26
2016    26
2017    26
2018    26
2019    26
2020    26
2021    26
2022    26
2023    26
2024    26
Name: count, dtype: int64

📌 Export the climate file for forward analysis.

In [17]:
df_crop_country_climate.to_csv(f'{processed_dir}/df_crop_country_climate.csv', index=False)
print(f"Exported: {processed_dir}/df_crop_country_climate.csv")

Exported: ../data/processed/df_crop_country_climate.csv


---